# **Project 1 - Naive Bayes and Logistic Regression for Text Classification**
In this project, you will build simple machine learning models to detect spam emails. You will implement and compare two types of Naive Bayes (Multinomial Naive Bayes on Bag-of-Words features and Bernoulli Naive Bayes on presence/absence features) and Logistic Regression.

CS 6375.001 || Anil Lingala (akl180001)

# Setup

In [107]:
# Imports
import sys
import os, csv, re
import string
from collections import Counter
from typing import List, Tuple, Dict
import numpy as np
from glob import glob

In [108]:
# Function to import NTLK and download punkt_tab and stopwords
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

def nltk_setup():
    nltk.download('punkt', quiet=True)
    nltk.download('stopwords', quiet=True)
    nltk.download('punkt_tab', quiet=True)

nltk_setup()
stop_words = set(stopwords.words('english'))

# Import Datasets:

There are three datasets: enron1, enron2, and enron4.
You have to perform the experiments described below on all the three datasets. Each data
set is divided into two sets: training and test. Each of them has two directories: spam and
ham. All files in the spam and ham folders are spam and ham messages respectively.

In [109]:
# Mount to google drive and datasets directory. Will print the directory structure.
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

BASE_DIR = "/content/drive/MyDrive/Colab Notebooks/Project 1/datasets"

!apt-get -qq install tree >/dev/null 2>&1
!bash -lc 'tree -L 3 "/content/drive/MyDrive/Colab Notebooks/Project 1/datasets" || true'

Mounted at /content/drive
/content/drive/MyDrive/Colab Notebooks/Project 1/datasets
├── enron1_bernoulli_test.csv
├── enron1_bernoulli_train.csv
├── enron1_bow_test.csv
├── enron1_bow_train.csv
├── enron1_test
│   └── test
│       ├── ham
│       └── spam
├── enron1_train
│   └── train
│       ├── ham
│       └── spam
├── enron2_bernoulli_test.csv
├── enron2_bernoulli_train.csv
├── enron2_bow_test.csv
├── enron2_bow_train.csv
├── enron2_test
│   └── test
│       ├── ham
│       └── spam
├── enron2_train
│   └── train
│       ├── ham
│       └── spam
├── enron4_bernoulli_test.csv
├── enron4_bernoulli_train.csv
├── enron4_bow_test.csv
├── enron4_bow_train.csv
├── enron4_test
│   └── test
│       ├── ham
│       └── spam
└── enron4_train
    └── train
        ├── ham
        └── spam

24 directories, 12 files


# Step 1: Data Preparation
Your task is to transform a collection of emails into a structured numerical representation
in the form of a feature matrix, where:

• Columns represent features (i.e., words from a predefined vocabulary).

• Rows represent examples (i.e., individual emails).

• Each entry in the matrix quantifies the presence of a word (column) in a given email
(row).

# Building the Vocabulary:

• Collect all unique words from the training set to create a fixed vocabulary
(the
features). Do not use the test set for vocabulary construction.

• Convert all text to lowercase and remove punctuation to ensure consistency (use
the same rules for train and test).

• Ignore very common words (stopwords) such as ”the,” ”is,” and ”and” to reduce
noise. You may use any text processing library (e.g., NLTK) to help with tokenization and text preprocessing. However, be sure to cite any external tools used
in your report.



In [110]:
labels = [("ham", 0), ("spam", 1)]
token_len = 2
symbol_discard = str.maketrans({c: " " for c in string.punctuation})

# Function returns clean list of lowercase tokens without punctuation, unclean and stop words.
def preprocess(text: str) -> List[str]:
    text = text.lower().translate(symbol_discard)
    tokens = word_tokenize(text)
    out = []
    for tok in tokens:
        if not re.fullmatch(r"[a-z0-9]+", tok):
          continue
        if tok in stop_words:
          continue
        if len(tok) < token_len:
          continue
        out.append(tok)
    return out

# Function that reads emails
def read_email_file(path: str) -> str:
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        return f.read()

In [111]:
example_path = "/content/drive/MyDrive/Colab Notebooks/Project 1/datasets/enron1_train/train/ham/0020.1999-12-15.farmer.ham.txt"

before_preprocess = read_email_file(example_path)
after_preprocess = preprocess(before_preprocess)

#print("Example before Preprocess:\n", before_preprocess[:500], "\n")
print("Tokens after Preprocess: ", after_preprocess[:50])

#!ls '/content/drive/MyDrive/Colab Notebooks/Project 1/datasets/enron1_train/train/ham' | head

Tokens after Preprocess:  ['subject', 'meter', '1431', 'nov', '1999', 'daren', 'could', 'please', 'resolve', 'issue', 'howard', 'office', 'next', 'two', 'days', 'done', 'please', 'let', 'george', 'know', 'thanks', 'aimee', 'forwarded', 'aimee', 'lannou', 'hou', 'ect', '12', '15', '99', '01', '27', 'pm', 'howard', 'camp', '12', '15', '99', '01', '01', 'pm', 'aimee', 'lannou', 'hou', 'ect', 'ect', 'cc', 'daren', 'farmer', 'hou']


In [112]:
# Function that reads and cleans every email in the given split. returns tokenized text plus labels (ham or spam).
def load_split(split_root: str):
    tokens_list, hs = [], []

    for path_name, label in labels:
        files = sorted(glob(os.path.join(split_root, path_name, "*")))
        for fpath in files:
            if not os.path.isfile(fpath):
              continue
            raw = read_email_file(fpath)
            tokens = preprocess(raw)
            tokens_list.append(tokens)
            hs.append(label)
    return tokens_list, hs

In [113]:
# Function that counts word frequency and builds vocab
def build_vocab(train_tokens: List[List[str]]) -> List[str]:
    countFreq = Counter()

    for token in train_tokens:
      countFreq.update(token)

    # sort token - priority
    #1) descending freq count
    #2) ascending word
    def sort_token(item):
        word, count = item
        return -count, word

    sorted_items = sorted(countFreq.items(), key=sort_token)
    return [w for w,_ in sorted_items]

# Generating Feature Matrices for Each Representation:

# Bag of Words:

– For each email, count how many times each word from the vocabulary appears.

– Store these counts in a row of the feature matrix.


In [114]:
# Function that converts each email tokens into a bag of words count matrix
def vectorize_bow(examples, vocab_index):
    emailCount, wordCount = len(examples), len(vocab_index)
    matrixCount = np.zeros((emailCount,wordCount), dtype=np.int32)

    for i, token_list in enumerate(examples):
        for w in token_list:
            j = vocab_index.get(w)
            if j is not None:
              matrixCount[i,j] += 1
    return matrixCount

# Bernoulli Representation:

– For each email, mark each word’s presence as 1 if it appears at least once and
0 otherwise.

– Store this binary information in a row of the feature matrix.

In [115]:
# Function that converts each email tokens into a Bernoulli matrix
def vectorize_bernoulli(examples, vocab_index):
    emailCount, wordCount = len(examples), len(vocab_index)
    matrixCount = np.zeros((emailCount,wordCount), dtype=np.int8)

    for i, token in enumerate(examples):
        seen = set()
        for w in token:
            j = vocab_index.get(w)
            if j is not None and j not in seen: #presence/absence
                matrixCount[i,j] = 1;
                seen.add(j)
    return matrixCount

#Storing the Datasets in CSV Format:

By applying these steps, you will create a
total of 12 datasets (3 datasets × 2 representations × train/test):

• 6 training sets (one BoW and one Bernoulli per dataset).

• 6 test sets (one BoW and one Bernoulli per dataset).

Each dataset should be stored as a CSV (Comma-Separated Values) file to ensure
compatibility with machine learning libraries.

In [116]:
# Function to save feature matrix and email labels to a CSV file
def save_csv(file_path, feature_matrix, email_labels, vocabulary):
    os.makedirs(os.path.dirname(file_path) or ".", exist_ok=True)
    with open(file_path, "w", newline="", encoding="utf-8") as csv_file:
        writer = csv.writer(csv_file)
        writer.writerow(vocabulary + ["label"])

        for row_index in range(feature_matrix.shape[0]):
            row_values = feature_matrix[row_index].tolist()
            label_value = int(email_labels[row_index])
            writer.writerow(row_values + [label_value])

# Function that returns (dataset_name, train_path, test_path) tuples.
def find_dataset_splits(datasets_root: str):
    subdirectories = [d for d in glob(os.path.join(datasets_root, "*")) if os.path.isdir(d)]

    train_folders = {}
    test_folders = {}

    for folder_path in subdirectories:
        folder_name = os.path.basename(folder_path)

        if folder_name.endswith("_train"):
            dataset_name = folder_name[:-6]
            train_folders[dataset_name] = os.path.join(folder_path, "train")

        elif folder_name.endswith("_test"):
            dataset_name = folder_name[:-5]
            test_folders[dataset_name] = os.path.join(folder_path, "test")

    dataset_splits = []
    for dataset_name in sorted(set(train_folders.keys()) & set(test_folders.keys())):
        dataset_splits.append(
            (dataset_name, train_folders[dataset_name], test_folders[dataset_name])
        )

    return dataset_splits

# Each dataset should follow the naming format:


```
# dataset_representation_set.csv
```


where:

• dataset: The dataset name (enron1, enron2, or enron4).

• representation: The text representation method:

– bow for the Bag of Words model.

– bernoulli for the Bernoulli model.


• set: The dataset split:

– train for the training set.

– test for the test set


In [117]:
# Function that saves matrices as CSV files
def generate_csv(dataset_name,train_split_path,test_split_path,output_directory):
    train_tokens,train_email_labels=load_split(train_split_path)
    vocabulary=build_vocab(train_tokens)

    vocab_index={word:i for i,word in enumerate(vocabulary)}

    bow_train_matrix=vectorize_bow(train_tokens,vocab_index)
    bernoulli_train_matrix=vectorize_bernoulli(train_tokens,vocab_index)

    test_tokens=load_split(test_split_path)
    test_email_labels=load_split(test_split_path)

    test_tokens, test_email_labels = load_split(test_split_path)
    bow_test_matrix = vectorize_bow(test_tokens, vocab_index)
    bernoulli_test_matrix = vectorize_bernoulli(test_tokens, vocab_index)

    save_csv(os.path.join(output_directory,f"{dataset_name}_bow_train.csv"),bow_train_matrix,train_email_labels,vocabulary)
    save_csv(os.path.join(output_directory,f"{dataset_name}_bow_test.csv"),bow_test_matrix,test_email_labels,vocabulary)
    save_csv(os.path.join(output_directory,f"{dataset_name}_bernoulli_train.csv"),bernoulli_train_matrix,train_email_labels,vocabulary)
    save_csv(os.path.join(output_directory,f"{dataset_name}_bernoulli_test.csv"),bernoulli_test_matrix,test_email_labels,vocabulary)
    print(f"[{dataset_name}] |V|={len(vocabulary)}  train={len(train_email_labels)}  test={len(test_email_labels)} → CSVs written")

#call generate_csv and print dataset counts
print("DATASETS_ROOT =",BASE_DIR)
dataset_splits=find_dataset_splits(BASE_DIR)

for dataset_name,train_split_path,test_split_path in dataset_splits:
    print(f"Dataset {dataset_name}: ")
    generate_csv(dataset_name,train_split_path,test_split_path,BASE_DIR)

DATASETS_ROOT = /content/drive/MyDrive/Colab Notebooks/Project 1/datasets
Dataset enron1: 
[enron1] |V|=9891  train=450  test=456 → CSVs written
Dataset enron2: 
[enron2] |V|=10242  train=463  test=478 → CSVs written
Dataset enron4: 
[enron4] |V|=17656  train=535  test=543 → CSVs written


# Step 2: Multinomial Naive Bayes

Implementation Details:

• Apply add-one Laplace smoothing (α = 1) to handle zero probabilities.

• Perform all probability calculations in log-space to prevent numerical underflow. Note
that, to avoid underflow during test time prediction, do not exponentiate the logprobabilities.

• Train your model using the Bag of Words training datasets and evaluate its performance on the corresponding test datasets.
After training the model, report the accuracy, precision, recall, and F1-score on
the test set.

Important: Use only the datasets generated using the Bag of Words approach for this
part.

In [118]:
# Function that loads bag of words csvs and returns feature matrix and labels
def load_bow_csv(csv_file):
    with open(csv_file, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        header = next(reader)
        vocab_words = header[:-1]

        feature_rows = []
        label_rows = []

        for row in reader:
            *words, label = row
            word_counts = [int(v) for v in words]
            feature_rows.append(word_counts)
            label_rows.append(int(label))

    features = np.array(feature_rows, dtype=np.int32)
    labels = np.asarray(label_rows, dtype=np.int32)
    #print("shape", features.shape, labels.shape)
    #print("vocab", len(vocab_words))
    return features, labels, vocab_words

tmp_x, tmp_y, tmp_v = load_bow_csv(os.path.join(BASE_DIR,"enron1_bow_train.csv"))
#print("test comb:", tmpX.shape, tmpy.shape)

In [119]:
# Function that trains multinomial NB with Laplace smoothing
def train_multinomial_nb(train_features, train_labels, alpha=1.0):
    classes = np.unique(train_labels)
    n_classes = len(classes)

    vocab_size = train_features.shape[1]
    doc_counts = np.bincount(train_labels, minlength=classes.max()+1).astype(float)
    docs_per_class = np.array([doc_counts[c] for c in classes], dtype=float)
    total = len(train_labels)
    priors = docs_per_class / total
    log_priors = np.log(priors)

    wc_per_class = np.zeros((n_classes, vocab_size), dtype=float)
    for i, c in enumerate(classes):
        rows = train_features[train_labels == c]
        wc_per_class[i] = rows.sum(axis=0)
        # print("test\n")
          #print(wc_per_class[i][:5])

    totals = wc_per_class.sum(axis=1)
    smoothed = (wc_per_class + alpha) / (totals[:, None] + alpha * vocab_size)
    log_likelihoods = np.log(smoothed)

    return {"classes": classes, "log_priors": log_priors, "log_likelihoods": log_likelihoods}

In [120]:
# Function that predicts classes using trained Multinomial Naive Bayes model
def predict_multinomial_nb(model, test_features):
    scores = test_features @ model["log_likelihoods"].T
    scores += model["log_priors"]

    class_indices = scores.argmax(axis=1)
    return model["classes"][class_indices]

In [121]:
# Function that computes accuracy, precision, recall, and F1 for binary classification
def metrics_binary(true_labels, predicted_labels, positive_class=1):
    true_labels = np.asarray(true_labels)
    predicted_labels = np.asarray(predicted_labels)

    tp = np.sum((true_labels == positive_class) & (predicted_labels == positive_class))
    fp = np.sum((true_labels != positive_class) & (predicted_labels == positive_class))
    fn = np.sum((true_labels == positive_class) & (predicted_labels != positive_class))

    accuracy = np.mean(true_labels == predicted_labels)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0

    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return accuracy, precision, recall, f1

In [122]:
# Function that trains and evaluates Multinomial nb on one dataset bow csv
def evaluate_dataset_mnb_bow(dataset_name, alpha=1.0, positive_class=1):
    train_csv = os.path.join(BASE_DIR, f"{dataset_name}_bow_train.csv")
    test_csv = os.path.join(BASE_DIR, f"{dataset_name}_bow_test.csv")

    row_train, col_train, _ = load_bow_csv(train_csv)
    row_test, col_test, _ = load_bow_csv(test_csv)

    model = train_multinomial_nb(row_train, col_train, alpha=alpha)
    predictions = predict_multinomial_nb(model, row_test)

    acc, prec, rec, f1 = metrics_binary(col_test, predictions, positive_class=positive_class)
   # print(f"[{dataset_name} | Multinomial NB] acc={acc:.4f} prec={prec:.4f} rec={rec:.4f} f1={f1:.4f}")
    return acc, prec, rec, f1

results = {}
for ds in ["enron1", "enron2", "enron4"]:
    results[ds] = evaluate_dataset_mnb_bow(ds, alpha=1.0, positive_class=1)

#results

In [123]:
## Function that pretty-prints Multinomial NB results
def print_multinomial_results(results_dict, digits=4):
    print("Multinomial NB \n")
    print("dataset    accuracy    precision    recall    f1")
    print("------------------------------------------------")

    total_acc = 0.0
    total_prec = 0.0
    total_rec = 0.0
    total_f1 = 0.0
    n = len(results_dict)

    for ds, (acc, prec, rec, f1) in results_dict.items():
        print(f"{ds:10} {acc:.{digits}f}    {prec:.{digits}f}    {rec:.{digits}f}    {f1:.{digits}f}")
        total_acc += acc
        total_prec += prec
        total_rec += rec
        total_f1 += f1

    print("------------------------------------------------")

print_multinomial_results(results)


Multinomial NB 

dataset    accuracy    precision    recall    f1
------------------------------------------------
enron1     0.9364    0.9348    0.8658    0.8990
enron2     0.9414    0.9322    0.8462    0.8871
enron4     0.9705    0.9699    0.9898    0.9797
------------------------------------------------


# Step 3: Bernoulli Naive Bayes

Implement the Bernoulli Naive Bayes algorithm (also known as Binary/Discrete Naive
Bayes), which models each word’s presence or absence in a document.

Implementation Details:

• Use add-one Laplace smoothing to avoid zero probabilities.

• Perform all computations in log-space to prevent underflow.

• Train your model using the Bernoulli training datasets and evaluate its performance
on the corresponding test datasets.

After training, report the accuracy, precision, recall, and F1-score on the test set. See
section on “Evaluation Metrics” at the end of this document on how to compute each of
these scores.

Important: Use only the datasets generated using the Bernoulli approach for this part.

In [124]:
# Function that loads Bernoulli CSVs and returns feature matrix and labels
def load_bernoulli_csv(file_path):

    with open(file_path, "r", encoding="utf-8") as f:
        rdr = csv.reader(f)
        header = next(rdr)
        vocab = header[:-1]

        feats, lbls = [], []
        for row in rdr:
            *feat, lbl = row
            feats.append([int(v) for v in feat])
            lbls.append(int(lbl))

    X = np.asarray(feats, dtype=np.int8)
    y = np.asarray(lbls, dtype=np.int32)

    return X, y, vocab


In [125]:
# Function that trains Bernoulli Naive Bayes with Laplace smoothing
def train_bernoulli_nb(train_data, train_targets, alpha=1.0):

    classes = np.unique(train_targets)
    n_classes = classes.size
    vocab_size = train_data.shape[1]

    class_counts = np.array([(train_targets == c).sum() for c in classes], dtype=np.float64)
    log_priors = np.log(class_counts / train_targets.size)

    word_counts = np.zeros((n_classes, vocab_size), dtype=np.float64)
    for i, c in enumerate(classes):
        word_counts[i] = (train_data[train_targets == c] > 0).sum(axis=0)
        # print("class", c, "sum:", word_counts[i][:5])

    probs = (word_counts + alpha) / (class_counts[:, None] + 2*alpha)
    log_probs = np.log(probs)
    log_not_probs = np.log(1.0 - probs)

    return {
        "classes": classes,
        "log_priors": log_priors,
        "log_theta": log_probs,
        "log_one_minus_theta": log_not_probs
    }


In [126]:
# Function that predicts classes using trained Bernoulli Naive Bayes model
def predict_bernoulli_nb(model,test_features):
    # score = log P(c) + x·logθ + (1−x)·log(1−θ)
    scores = (test_features @ model["log_theta"].T) + ((1-test_features) @ model["log_one_minus_theta"].T)

    scores += model["log_priors"]

    class_indices=scores.argmax(axis=1)
    return model["classes"][class_indices]

In [127]:
# Function that computes accuracy, precision, recall, and F1 for binary classification
def metrics_binary(true_labels,predicted_labels,positive_class=1):
    true_labels=np.asarray(true_labels); predicted_labels=np.asarray(predicted_labels)

    tp=np.sum((true_labels==positive_class)&(predicted_labels==positive_class))
    fp=np.sum((true_labels!=positive_class)&(predicted_labels==positive_class))
    fn=np.sum((true_labels==positive_class)&(predicted_labels!=positive_class))

    accuracy=np.mean(true_labels==predicted_labels)
    precision=tp/(tp+fp) if (tp+fp)>0 else 0.0
    recall=tp/(tp+fn) if (tp+fn)>0 else 0.0

    f1=2*precision*recall/(precision+recall) if (precision+recall)>0 else 0.0

    return accuracy,precision,recall,f1

In [128]:
# Function that trains and evaluates Bernoulli NB on one dataset’s Bernoulli CSVs
def evaluate_dataset_bnb_bernoulli(dataset_name,alpha=1.0,positive_class=1):
    train_csv=os.path.join(BASE_DIR,f"{dataset_name}_bernoulli_train.csv")
    test_csv =os.path.join(BASE_DIR,f"{dataset_name}_bernoulli_test.csv")

    X_train,y_train,_=load_bernoulli_csv(train_csv)
    X_test ,y_test ,_=load_bernoulli_csv(test_csv)

    model=train_bernoulli_nb(X_train,y_train,alpha=alpha)
    predictions=predict_bernoulli_nb(model,X_test)

    acc,prec,rec,f1=metrics_binary(y_test,predictions,positive_class=positive_class)
    #print(f"[{dataset_name} | Bernoulli NB] acc={acc:.4f} prec={prec:.4f} rec={rec:.4f} f1={f1:.4f}")
    return acc,prec,rec,f1

bernoulli_results={}
for ds in ["enron1","enron2","enron4"]:
    bernoulli_results[ds]=evaluate_dataset_bnb_bernoulli(ds,alpha=1.0,positive_class=1)

#bernoulli_results

In [129]:
# Function that pretty-prints Bernoulli NB results
def print_bernoulli_results(results_dict, digits=4):
    print("Bernoulli NB \n")
    print("dataset    accuracy    precision    recall    f1")
    print("------------------------------------------------")

    total_acc = 0.0
    total_prec = 0.0
    total_rec = 0.0
    total_f1 = 0.0
    n = len(results_dict)

    for ds, (acc, prec, rec, f1) in results_dict.items():
        print(f"{ds:10} {acc:.{digits}f}    {prec:.{digits}f}    {rec:.{digits}f}    {f1:.{digits}f}")
        total_acc += acc
        total_prec += prec
        total_rec += rec
        total_f1 += f1
        #print("------------------------------------------------")
    print("------------------------------------------------")
print_bernoulli_results(bernoulli_results)

Bernoulli NB 

dataset    accuracy    precision    recall    f1
------------------------------------------------
enron1     0.7303    0.8824    0.2013    0.3279
enron2     0.7741    0.8929    0.1923    0.3165
enron4     0.9208    0.9009    1.0000    0.9479
------------------------------------------------


# Step 4 - Logistic Regression

Implement the Maximum Conditional A Posteriori (MCAP) Logistic Regression
algorithm with ℓ2 regularization (equivalently, LR with a Gaussian prior on weights). See
the supplementary notes provided with this project on implementing logistic regression efficiently.

Implementation details


*   Use gradient ascent to learn the model parameters.
*   Perform hyperparameter tuning by selecting an optimal λ value for ℓ2 regularization.
*   Split the training data into 70% training and 30% validation for tuning λ (use the
training split only; do not use the test set for tuning).
*   After selecting the best λ, train the model on the full training set.
*   Report accuracy, precision, recall, and F1-score on the test set (classify as spam
if P(y=1 | x) ≥ 0.5).

Since Logistic Regression can handle both feature representations, use both the Bag of Words and Bernoulli datasets.

Important: Ensure a suitable learning rate (e.g. 0.001 or 0.01) to avoid slow convergence
or divergence. Set a hard limit on the number of iterations (e.g. 500 iterations) to control
runtime.








In [130]:
# Function that loads a CSV X,y, vocab
def load_feature_csv(csv_path):
    with open(csv_path, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        header = next(reader)
        vocab_words = header[:-1]

        feature_rows, label_rows = [], []
        for row in reader:
            *feat, label = row
            feature_rows.append([int(v) for v in feat])
            label_rows.append(int(label))

    features = np.asarray(feature_rows, dtype=np.float64)
    labels   = np.asarray(label_rows, dtype=np.int32)
    return features, labels, vocab_words

In [131]:
# Function that splits X,y into train and val
def split_train_validation(features, labels, val_ratio=0.30, seed=42):
    rng = np.random.default_rng(seed)
    total = features.shape[0]
    idx = np.arange(total)
    rng.shuffle(idx)

    cut = int(round(total * (1.0 - val_ratio)))
    train_idx, valid_idx = idx[:cut], idx[cut:]

    return features[train_idx], labels[train_idx], features[valid_idx], labels[valid_idx]

In [132]:
# sigmoid with clipping to avoid overflow
def sigmoid(z):
    z = np.clip(z, -709, 709)
    return 1.0 / (1.0 + np.exp(-z))

In [133]:
# train logistic regression with gradient ascent and l2 penalty
def train_logistic_regression(train_features, train_labels, lr=0.01, iters=500, lam=1.0):
    num_samples, num_features = train_features.shape

    design = np.hstack([np.ones((num_samples, 1), dtype=np.float64), train_features])

    labels = train_labels.astype(np.float64)
    weights = np.zeros(num_features + 1, dtype=np.float64)

    for _ in range(iters):
        z = design @ weights
        probs = sigmoid(z)
        errors = labels - probs
        grad = design.T @ errors
        grad[1:] -= lam * weights[1:]
        weights += lr * grad
        #print(probs + " " + errors + " " + grad + weights)
    return weights

In [134]:
# tune lambda by splitting into 70/30 train valid
def tune_lambda(train_features, train_labels, lambda_list, lr=0.01, iters=500):
    tr_features, tr_labels, va_features, va_labels = split_train_validation(train_features, train_labels)

    best_lambda = None
    best_f1 = -1.0

    for lam in lambda_list:
        weights = train_logistic_regression(tr_features, tr_labels, lr=lr, iters=iters, lam=lam)
        preds = predict_labels(weights, va_features, threshold=0.5)
        acc, prec, rec, f1 = metrics_binary(va_labels, preds, positive_class=1)

        if f1 > best_f1:
            best_f1, best_lambda = f1, lam

    return best_lambda

In [135]:
# retrain full model with chosen lambda
def final_train(train_features, train_labels, best_lambda, lr=0.01, iters=500):
    return train_logistic_regression(train_features, train_labels, lr=lr, iters=iters, lam=best_lambda)

In [136]:
# predict probabilities given weights
def predict_proba(weights, features):
    num_samples = features.shape[0]
    design = np.hstack([np.ones((num_samples, 1), dtype=np.float64), features])
    return sigmoid(design @ weights)

In [137]:
# predict class labels (0 or 1) at threshold
def predict_labels(weights, features, threshold=0.5):
    probs = predict_proba(weights, features)
    return (probs >= threshold).astype(np.int32)


In [138]:
# compute accuracy, precision, recall, f1
def metrics_binary(true_labels, predicted_labels, positive_class=1):
    true_labels = np.asarray(true_labels)
    predicted_labels = np.asarray(predicted_labels)

    tp = np.sum((true_labels == positive_class) & (predicted_labels == positive_class))
    fp = np.sum((true_labels != positive_class) & (predicted_labels == positive_class))
    fn = np.sum((true_labels == positive_class) & (predicted_labels != positive_class))

    acc = np.mean(true_labels == predicted_labels) if true_labels.size else 0.0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0

    f1   = 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0
    return acc, prec, rec, f1

In [139]:
# evaluate logistic regression on dataset+representation
def evaluate_logreg_dataset(dataset_name, representation, lambda_list, lr=0.01, iters=500, positive_class=1):
    train_csv = os.path.join(BASE_DIR, f"{dataset_name}_{representation}_train.csv")
    test_csv  = os.path.join(BASE_DIR, f"{dataset_name}_{representation}_test.csv")

    train_features, train_labels, _ = load_feature_csv(train_csv)
    test_features, test_labels, _   = load_feature_csv(test_csv)

    best_lambda = tune_lambda(train_features, train_labels, lambda_list, lr=lr, iters=iters)
    weights = final_train(train_features, train_labels, best_lambda, lr=lr, iters=iters)

    preds = predict_labels(weights, test_features, threshold=0.5)
    acc, prec, rec, f1 = metrics_binary(test_labels, preds, positive_class=positive_class)

    #print(f"[{dataset_name} | LR-{representation}] λ*={best_lambda} acc={acc:.4f} prec={prec:.4f} rec={rec:.4f} f1={f1:.4f}")
    return {"lambda": best_lambda, "acc": acc, "prec": prec, "rec": rec, "f1": f1}

In [140]:
# run LR on all datasets for BoW + Bernoulli
def run_logreg_all(lambda_list=(0.01, 0.1, 1.0), lr=0.01, iters=500):
    results = {"bow": {}, "bernoulli": {}}
    for ds in ["enron1", "enron2", "enron4"]:
        results["bow"][ds] = evaluate_logreg_dataset(ds, "bow", lambda_list, lr=lr, iters=iters)
    for ds in ["enron1", "enron2", "enron4"]:
        results["bernoulli"][ds] = evaluate_logreg_dataset(ds, "bernoulli", lambda_list, lr=lr, iters=iters)
    return results

In [143]:
# print logistic regression results nicely
def print_logreg_results(results_dict, digits=4):
    for rep in ["bow", "bernoulli"]:
        print(f"\nLogistic Regression ({rep})\n")
        print("dataset    lambda    accuracy    precision    recall    f1")
        print("-------------------------------------------------------------")

        total_acc = 0.0
        total_prec = 0.0
        total_rec = 0.0
        total_f1 = 0.0
        n = 0

        for ds, metrics in results_dict[rep].items():
            lam = metrics["lambda"]
            acc, prec, rec, f1 = metrics["acc"], metrics["prec"], metrics["rec"], metrics["f1"]
            print(f"{ds:10} {lam:.4f}   {acc:.{digits}f}    {prec:.{digits}f}    {rec:.{digits}f}    {f1:.{digits}f}")
            total_acc  += acc
            total_prec += prec
            total_rec  += rec
            total_f1   += f1
            n += 1

        print("-------------------------------------------------------------")
        print(f"{'average':10} ----   {total_acc/n:.{digits}f}    {total_prec/n:.{digits}f}    {total_rec/n:.{digits}f}    {total_f1/n:.{digits}f}")


In [144]:
#run
logreg_results = run_logreg_all(lambda_list=(0.01, 0.1, 1.0), lr=0.01, iters=500)
print_logreg_results(logreg_results)


Logistic Regression (bow)

dataset    lambda    accuracy    precision    recall    f1
-------------------------------------------------------------
enron1     0.0100   0.9671    0.9241    0.9799    0.9511
enron2     0.0100   0.9540    0.8971    0.9385    0.9173
enron4     0.0100   0.9705    0.9607    1.0000    0.9799
-------------------------------------------------------------
average    ----   0.9639    0.9273    0.9728    0.9495

Logistic Regression (bernoulli)

dataset    lambda    accuracy    precision    recall    f1
-------------------------------------------------------------
enron1     0.0100   0.9605    0.9226    0.9597    0.9408
enron2     1.0000   0.9498    0.9344    0.8769    0.9048
enron4     0.0100   0.9705    0.9607    1.0000    0.9799
-------------------------------------------------------------
average    ----   0.9603    0.9392    0.9456    0.9418
